In [1]:
# Install Streamlit
!pip install streamlit pyngrok

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 15.1 MB/s eta 0:00:00


In [8]:
!pip install -q langchain
!pip install -q langchain-community
!pip install -q langchain-google-genai
!pip install -q chromadb
!pip install -q pypdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 9.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 52.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.7/61.7 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 6.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.5/70.5 kB 1.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 811.4 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 43.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 22.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 68.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━

In [28]:
import os

try:
    import streamlit as st
    api_key = st.secrets["GOOGLE_API_KEY"]

except:
    from google.colab import userdata
    api_key = userdata.get("GOOGLE_API_KEY")

os.environ["GOOGLE_API_KEY"] = api_key

In [30]:
from google.colab import userdata
import os

os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")

In [48]:
# %%writefile requirements.txt
# streamlit
# langchain
# langchain-community
# langchain-google-genai
# chromadb
# pypdf

In [50]:
%%writefile requirements.txt
streamlit
langchain>=0.3.0
langchain-community>=0.3.0
langchain-google-genai>=2.0.0
chromadb>=0.5.0
pypdf>=5.0.0
google-generativeai>=0.8.0

Overwriting requirements.txt


Upload PDF

In [5]:
# Upload multiple PDFs
from google.colab import files

uploaded = files.upload()

# Show uploaded files
print("Uploaded PDFs:")
for file_name in uploaded.keys():
    print(file_name)

Saving langchain_lecture3_models_notes.pdf to langchain_lecture3_models_notes.pdf
Saving langchain_lecture4_prompts_notes.pdf to langchain_lecture4_prompts_notes (1).pdf
Uploaded PDFs:
langchain_lecture3_models_notes.pdf
langchain_lecture4_prompts_notes (1).pdf


In [39]:
# from google.colab import files

# uploaded = files.upload()

Load PDF

In [6]:
# Import PDF loader
from langchain_community.document_loaders import PyPDFLoader

# List to store all pages from all PDFs
documents = []

# Loop through each uploaded PDF
for pdf_file in uploaded.keys():

    print(f"Loading {pdf_file} ...")

    # Create loader object
    loader = PyPDFLoader(pdf_file)

    # Load pages from current PDF
    docs = loader.load()

    # Add source filename to metadata
    for doc in docs:
        doc.metadata["source_file"] = pdf_file

    # Add pages to master document list
    documents.extend(docs)

print(f"\nTotal Pages Loaded: {len(documents)}")

Loading langchain_lecture3_models_notes.pdf ...
Loading langchain_lecture4_prompts_notes (1).pdf ...

Total Pages Loaded: 42


In [41]:
# from langchain_community.document_loaders import PyPDFLoader

# pdf_path = list(uploaded.keys())[0]

# loader = PyPDFLoader(pdf_path)

# documents = loader.load()

# print("Total Pages:", len(documents))

In [32]:
# Import LangChain's text splitter
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Create a text splitter object
text_splitter = RecursiveCharacterTextSplitter(

    # Maximum size of each chunk (in characters)
    chunk_size=2100,

    # Number of overlapping characters between consecutive chunks
    # Helps preserve context across chunk boundaries
    chunk_overlap=200
)

# Split the PDF pages into smaller chunks
docs = text_splitter.split_documents(documents)

# Print total number of chunks created
print("Number of chunks:", len(docs))

Number of chunks: 47


In [33]:
# Import Gemini embedding model
from langchain_google_genai import GoogleGenerativeAIEmbeddings

# Initialize embedding model
# This model converts text into numerical vectors
embeddings = GoogleGenerativeAIEmbeddings(
    model="models/gemini-embedding-001"
)

In [34]:
# Import Chroma vector database
from langchain_community.vectorstores import Chroma

# Convert chunks into embeddings and store them
vectorstore = Chroma.from_documents(

    # Text chunks generated in previous step
    documents=docs,

    # Embedding model used to convert text to vectors
    embedding=embeddings
)

print("Vector database created successfully.")

Vector database created successfully.


In [35]:
# Convert vector database into a retriever object
retriever = vectorstore.as_retriever(

    # Return top 3 most relevant chunks for each query
    search_kwargs={"k": 3}
)

print("Retriever created successfully.")

Retriever created successfully.


In [36]:
# Import Gemini chat model
from langchain_google_genai import ChatGoogleGenerativeAI

# Initialize Gemini model
llm = ChatGoogleGenerativeAI(

    # Gemini model name
    model="gemini-2.5-flash",

    # Temperature = 0 means deterministic answers
    # Useful for factual RAG applications
    temperature=0
)

print("LLM initialized successfully.")

LLM initialized successfully.


In [37]:
# User question
query = "What is this document about?"

# Retrieve top relevant chunks from vector database
retrieved_docs = retriever.invoke(query)

# Display retrieved chunks
for i, doc in enumerate(retrieved_docs):

    print(f"\n---------- Chunk {i+1} ----------")

    # Print first 500 characters of retrieved chunk
    print(doc.page_content[:500])


---------- Chunk 1 ----------
Concept Definition WhyItExists
Internal
Working UseCases Limitations
Cosine
Similarity
Measureof
theangle
betweentwo
vectors
(similarity
score)
Standard
metricto
compare
embedding
vectors
cosine_similarity(v1,
v2)from
scikit-learn;
expects2D
inputs
Semantic
search,docu-
ment/query
matching,
RAGretrieval
Only
captures
geometric
closenessin
embedding
space,not
factual
correctness
5. CodeNotes
CodeBlock Purpose
Key
Classes/Functions
Key
Parameters
Common
Mistakes
Production
BestPractice
OpenAILLM
c

---------- Chunk 2 ----------
Concept Definition WhyItExists
Internal
Working UseCases Limitations
Cosine
Similarity
Measureof
theangle
betweentwo
vectors
(similarity
score)
Standard
metricto
compare
embedding
vectors
cosine_similarity(v1,
v2)from
scikit-learn;
expects2D
inputs
Semantic
search,docu-
ment/query
matching,
RAGretrieval
Only
captures
geometric
closenessin
embedding
space,not
factual
correctness
5. CodeNotes
CodeBlock Purpose
Key
Classes/Functions
Key

In [38]:
# Import prompt template
from langchain_core.prompts import PromptTemplate

# Define prompt for RAG
prompt = PromptTemplate(
    input_variables=["context", "question"],

    template="""
You are a helpful assistant.

Answer the question using ONLY the provided context.

If the answer is not present in the context, say:
"I could not find this information in the document."

Context:
{context}

Question:
{question}

Answer:
"""
)

In [39]:
# Function to ask questions to the PDF
def ask_pdf(question):

    # Retrieve relevant chunks from vector database
    retrieved_docs = retriever.invoke(question)

    # Combine retrieved chunks into a single context string
    context = "\n\n".join(
        [doc.page_content for doc in retrieved_docs]
    )

    # Create final prompt for Gemini
    final_prompt = prompt.format(
        context=context,
        question=question
    )

    # Generate answer using Gemini
    response = llm.invoke(final_prompt)

    # Print answer
    print("\nAnswer:")
    print(response.content)

improve

In [40]:
def ask_pdf(question):

    # Retrieve relevant chunks
    retrieved_docs = retriever.invoke(question)

    # Combine retrieved text
    context = "\n\n".join(
        [doc.page_content for doc in retrieved_docs]
    )

    # Create final prompt
    final_prompt = prompt.format(
        context=context,
        question=question
    )

    # Generate answer
    response = llm.invoke(final_prompt)

    print("\nAnswer:\n")
    print(response.content)

    print("\nSources Used:\n")

    # Remove duplicates
    sources_seen = set()

    for doc in retrieved_docs:

        source = (
            doc.metadata["source_file"],
            doc.metadata["page"] + 1
        )

        if source not in sources_seen:

            print(
                f"📄 {doc.metadata['source_file']} "
                f"(Page {doc.metadata['page'] + 1})"
            )

            sources_seen.add(source)

In [41]:
question = "Explain inventory optimization"

retrieved_docs = retriever.invoke(question)

for doc in retrieved_docs:
    print(
        f"{doc.metadata['source_file']} "
        f"- Page {doc.metadata['page'] + 1}"
    )

langchain_lecture3_models_notes.pdf - Page 16
langchain_lecture3_models_notes.pdf - Page 16
langchain_lecture3_models_notes.pdf - Page 14


In [42]:
ask_pdf("Summarize the document in 5 points.")


Answer:

Here is a 5-point summary of the document:

1.  The document defines "Retriever" as a component that fetches relevant vectors/documents for a query, which is important for the production version of a document-similarity demo.
2.  It compares LLMs and ChatModels, noting that ChatModels support multi-turn conversations, role assignment, and are recommended by LangChain for new projects, unlike LLMs which are String→String and lack these features.
3.  A comparison of Closed-Source vs Open-Source models highlights that closed-source involves pay-per-token API fees and sends data to provider servers, while open-source is free to self-host, offers full control/customization, keeps data local, but requires solid hardware.
4.  The document contrasts HuggingFace Inference API with Local Download, stating the API has low setup effort but sends data to HF servers, while local download has higher setup but is fully local and free after download, best for privacy-sensitive or offline prod

In [43]:
%%writefile app.py

# Import libraries
import streamlit as st
import os
import tempfile

from langchain_community.document_loaders import PyPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_google_genai import (
    GoogleGenerativeAIEmbeddings,
    ChatGoogleGenerativeAI
)
from langchain_community.vectorstores import Chroma


# Page configuration
st.set_page_config(
    page_title="Multi PDF Chatbot",
    page_icon="📚"
)

st.title("📚 Multi PDF Chatbot")

# API key input
api_key = st.text_input(
    "Enter Gemini API Key",
    type="password"
)

if api_key:
    os.environ["GOOGLE_API_KEY"] = api_key

# Upload PDFs
uploaded_files = st.file_uploader(
    "Upload PDFs",
    type="pdf",
    accept_multiple_files=True
)

# Process PDFs
if uploaded_files and api_key:

    documents = []

    for uploaded_file in uploaded_files:

        # Save temporary PDF
        with tempfile.NamedTemporaryFile(
            delete=False,
            suffix=".pdf"
        ) as tmp_file:

            tmp_file.write(uploaded_file.read())
            tmp_path = tmp_file.name

        # Load PDF
        loader = PyPDFLoader(tmp_path)
        docs = loader.load()

        # Store source filename
        for doc in docs:
            doc.metadata["source_file"] = uploaded_file.name

        documents.extend(docs)

    # Split documents
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=1000,
        chunk_overlap=200
    )

    chunks = splitter.split_documents(documents)

    # Embeddings
    embeddings = GoogleGenerativeAIEmbeddings(
        model="models/embedding-001"
    )

    # Vector DB
    vectorstore = Chroma.from_documents(
        chunks,
        embeddings
    )

    retriever = vectorstore.as_retriever(
        search_kwargs={"k": 3}
    )

    # Gemini model
    llm = ChatGoogleGenerativeAI(
        model="gemini-2.5-flash",
        temperature=0
    )

    # Question box
    question = st.text_input(
        "Ask your question"
    )

    if question:

        retrieved_docs = retriever.invoke(question)

        context = "\n\n".join(
            [doc.page_content for doc in retrieved_docs]
        )

        prompt = f"""
Answer using only the context below.

Context:
{context}

Question:
{question}

Answer:
"""

        response = llm.invoke(prompt)

        st.subheader("Answer")
        st.write(response.content)

        st.subheader("Sources")

        shown = set()

        for doc in retrieved_docs:

            source = (
                doc.metadata["source_file"],
                doc.metadata["page"]
            )

            if source not in shown:

                st.write(
                    f"📄 {doc.metadata['source_file']} "
                    f"(Page {doc.metadata['page'] + 1})"
                )

                shown.add(source)

Overwriting app.py


In [44]:
!ls

 app.py
 CampusX_LangChain_Lecture5_Structured_Output_Notes.pdf
 langchain_lecture3_models_notes.pdf
'langchain_lecture4_prompts_notes (1).pdf'
 langchain_lecture4_prompts_notes.pdf
 LangChain_Lecture6_OutputParsers_Notes.pdf
 logs.txt
 requirements.txt
 sample_data


In [45]:
from google.colab import files

files.download("app.py")
files.download("requirements.txt")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>